# Judge-Guided Rationale Training For e-SNLI

This notebook is tuned for the real objective: improve test-set label prediction, not just smoke-test the pipeline.

In [ ]:
from pathlib import Path
import os
import subprocess

candidates = [
    Path.cwd(),
    Path.cwd() / 'distilling-step-by-step',
    Path('/kaggle/working/distilling-step-by-step'),
]

REPO_ROOT = None
for candidate in candidates:
    if (candidate / 'run.py').exists() and (candidate / 'evaluate_test.py').exists():
        REPO_ROOT = candidate.resolve()
        break

if REPO_ROOT is None:
    raise FileNotFoundError('Could not find the repo root containing run.py and evaluate_test.py. Put the notebook inside the repo or place the repo at /kaggle/working/distilling-step-by-step.')

os.chdir(REPO_ROOT)
print('Using repo root:', REPO_ROOT)
try:
    print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
    print(subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())
except Exception as exc:
    print('Git metadata unavailable:', exc)


In [ ]:
!pwd
!ls -la | sed -n '1,60p'
!python - <<'PY'
from pathlib import Path
for rel in ['run.py', 'train_utils.py', 'evaluate_test.py', 'selection_utils.py']:
    p = Path(rel)
    print(rel, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else None)
PY

# Repo root confirmed above.

# Already running inside the repo root.

In [5]:
import sys
import torch
import transformers
import datasets

print("python", sys.version)
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu count", torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        print(f"gpu[{index}]", torch.cuda.get_device_name(index))


python 3.10.12 | packaged by conda-forge | (main, Jun 23 2023, 22:40:32) [GCC 12.3.0]
torch 2.0.0
transformers 4.30.2
datasets 2.1.0
cuda available True
gpu count 2
gpu[0] Tesla T4
gpu[1] Tesla T4


In [6]:
import os
import json
import shlex
import subprocess
from pathlib import Path

os.environ["WANDB_DISABLED"] = "true"

# Fast strong baseline: keep the new objective, but stay close to the old training cost.
SANITY_RUN = False
USE_MULTI_GPU = False
USE_FP16 = True
USE_GRADIENT_CHECKPOINTING = False
RUN_PRECHECK = True

TRAIN_SELECTION_POLICY = "thesis_prior"
TRAIN_NUM_RATIONALES = 1
LABEL_TYPE = "gt"
ALPHA = 0.90
MAX_INPUT_LENGTH = 512
GEN_MAX_LEN = 64

TRAIN_FILENAME = f"selected_{TRAIN_SELECTION_POLICY}_{TRAIN_NUM_RATIONALES}"
TRAIN_SELECTED_CSV = f"artifacts/esnli_selection/selected_{TRAIN_SELECTION_POLICY}_top{TRAIN_NUM_RATIONALES}.csv"
MODEL_PATH = Path("/kaggle/working/model_path") / TRAIN_FILENAME
TEST_DATA_PATH = "/kaggle/input/datasets/nightfury1103/anli-esnli-test/esnli_test.csv"
EVAL_OUTPUT_DIR = "artifacts/test_eval"
TRAIN_LOG_PATH = Path("artifacts") / "train_run.log"
EVAL_LOG_PATH = Path("artifacts") / "eval_run.log"
TRAIN_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

if SANITY_RUN:
    FROM_PRETRAINED = "t5-small"
    TRAIN_ARGS = {
        "batch_size": 2,
        "eval_batch_size": 2,
        "grad_steps": 1,
        "max_steps": 20,
        "eval_steps": 20,
    }
else:
    FROM_PRETRAINED = "google/t5-v1_1-base"
    TRAIN_ARGS = {
        "batch_size": 4,
        "eval_batch_size": 16,
        "grad_steps": 1,
        "max_steps": 1200,
        "eval_steps": 600,
    }

print("SANITY_RUN:", SANITY_RUN)
print("USE_MULTI_GPU:", USE_MULTI_GPU)
print("USE_FP16:", USE_FP16)
print("USE_GRADIENT_CHECKPOINTING:", USE_GRADIENT_CHECKPOINTING)
print("RUN_PRECHECK:", RUN_PRECHECK)
print("FROM_PRETRAINED:", FROM_PRETRAINED)
print("TRAIN_SELECTION_POLICY:", TRAIN_SELECTION_POLICY)
print("TRAIN_NUM_RATIONALES:", TRAIN_NUM_RATIONALES)
print("LABEL_TYPE:", LABEL_TYPE)
print("ALPHA:", ALPHA)
print("MAX_INPUT_LENGTH:", MAX_INPUT_LENGTH)
print("MODEL_PATH:", MODEL_PATH)
print("TRAIN_SELECTED_CSV:", TRAIN_SELECTED_CSV)
print("TRAIN_LOG_PATH:", TRAIN_LOG_PATH)


SANITY_RUN: False
USE_MULTI_GPU: False
USE_FP16: True
USE_GRADIENT_CHECKPOINTING: False
FROM_PRETRAINED: google/t5-v1_1-base
TRAIN_SELECTION_POLICY: thesis_prior
TRAIN_NUM_RATIONALES: 1
LABEL_TYPE: gt
ALPHA: 0.9
MAX_INPUT_LENGTH: 512
MODEL_PATH: /kaggle/working/model_path/selected_thesis_prior_1
TRAIN_SELECTED_CSV: artifacts/esnli_selection/selected_thesis_prior_top1.csv
TRAIN_LOG_PATH: artifacts/train_run.log


# Preflight

Run this tiny selected-rationale regression check before the expensive training run. It should finish with non-empty label predictions and exact accuracy `1.0`.


In [ ]:
if RUN_PRECHECK:
    precheck_cmd = [
        "python", "debug_selected_pipeline.py",
        "--max_steps", "20",
        "--eval_steps", "20",
        "--min_accuracy", "0.0",
    ]
    print("Running preflight:", " ".join(precheck_cmd))
    subprocess.run(precheck_cmd, check=True)
else:
    print("Skipping preflight. Set RUN_PRECHECK = True to enable it.")


In [7]:
prepare_cmd = [
    "python", "prepare_selected_rationales.py",
    "--output-dir", "artifacts/esnli_selection",
    "--selection-policy", TRAIN_SELECTION_POLICY,
    "--num-selected-rationales", str(TRAIN_NUM_RATIONALES),
]
print("Running:", " ".join(prepare_cmd))
subprocess.run(prepare_cmd, check=True)

import pandas as pd
selected_df = pd.read_csv(TRAIN_SELECTED_CSV)
print("split counts:", selected_df["split"].value_counts().to_dict())
selected_type_columns = [column for column in selected_df.columns if column.startswith("rationale_type_")]
for column in selected_type_columns:
    print(column, selected_df[column].value_counts().head().to_dict())
selected_df.head(3)


Running: python prepare_selected_rationales.py --output-dir artifacts/esnli_selection --selection-policy thesis_prior --num-selected-rationales 1


Canonical dataset saved to artifacts/esnli_selection/canonical_esnli_rationales.csv
Selected dataset saved to artifacts/esnli_selection/selected_thesis_prior_top1.csv
Report saved to artifacts/esnli_selection/selected_thesis_prior_top1_report.json
split counts: {'train': 7390, 'valid': 919, 'test': 911}
rationale_type_1 {'neutral': 8329, 'contrastive': 430, 'historical': 240, 'consensus': 79, 'comparative': 71}


,premise,hypothesis,input,label,split,pair_key,available_rationale_count,rationale_1,rationale_type_1,rationale_score_1,rationale_keep_1,rationale_selection_source_1
0,A person on a horse jumps over a broken down a...,A person is training his horse for a competition.,A person on a horse jumps over a broken down a...,neutral,test,a person on a horse jumps over a broken down a...,7,1. The correct answer is **neutral**. The prem...,neutral,5.0,True,thesis_prior
1,A person on a horse jumps over a broken down a...,"A person is at a diner, ordering an omelette.",A person on a horse jumps over a broken down a...,contradiction,test,a person on a horse jumps over a broken down a...,7,2. The correct answer is **contradiction**. Th...,neutral,5.0,True,thesis_prior
2,A person on a horse jumps over a broken down a...,"A person is outdoors, on a horse.",A person on a horse jumps over a broken down a...,entailment,train,a person on a horse jumps over a broken down a...,7,3. The correct answer is **entailment**. The p...,neutral,5.0,True,thesis_prior


In [8]:
train_cmd = [
    "python", "run.py",
    "--dataset", "esnli",
    "--selected_rationale_path", TRAIN_SELECTED_CSV,
    "--selection_policy", TRAIN_SELECTION_POLICY,
    "--num_selected_rationales", str(TRAIN_NUM_RATIONALES),
    "--model_type", "task_prefix",
    "--label_type", LABEL_TYPE,
    "--batch_size", str(TRAIN_ARGS["batch_size"]),
    "--eval_batch_size", str(TRAIN_ARGS["eval_batch_size"]),
    "--grad_steps", str(TRAIN_ARGS["grad_steps"]),
    "--max_input_length", str(MAX_INPUT_LENGTH),
    "--gen_max_len", str(GEN_MAX_LEN),
    "--max_steps", str(TRAIN_ARGS["max_steps"]),
    "--eval_steps", str(TRAIN_ARGS["eval_steps"]),
    "--from_pretrained", FROM_PRETRAINED,
    "--alpha", str(ALPHA),
]

if USE_FP16 and not SANITY_RUN:
    train_cmd.append("--fp16")
if USE_GRADIENT_CHECKPOINTING and not SANITY_RUN:
    train_cmd.append("--gradient_checkpointing")
if USE_MULTI_GPU and not SANITY_RUN:
    train_cmd.extend([
        "--multi_gpu",
        "--num_gpus", "2",
        "--dataloader_num_workers", "2",
        "--ddp_find_unused_parameters", "false",
    ])

train_cmd_str = " ".join(shlex.quote(part) for part in train_cmd)
wrapped_cmd = f"set -o pipefail; {train_cmd_str} 2>&1 | tee {shlex.quote(str(TRAIN_LOG_PATH))}"
print("Running:", wrapped_cmd)
subprocess.run(["bash", "-lc", wrapped_cmd], check=True)
assert MODEL_PATH.exists(), f"Expected saved model at {MODEL_PATH}"


Running: set -o pipefail; python run.py --dataset esnli --selected_rationale_path artifacts/esnli_selection/selected_thesis_prior_top1.csv --selection_policy thesis_prior --num_selected_rationales 1 --model_type task_prefix --label_type gt --batch_size 4 --eval_batch_size 16 --grad_steps 1 --max_input_length 512 --gen_max_len 64 --max_steps 1200 --eval_steps 600 --from_pretrained google/t5-v1_1-base --alpha 0.9 --fp16 2>&1 | tee artifacts/train_run.log
/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.s

# Test

Evaluate the exact checkpoint produced above on the held-out test set.

In [9]:
import pandas as pd
from IPython.display import display


In [10]:
eval_cmd = [
    "python", "evaluate_test.py",
    "--model_path", str(MODEL_PATH),
    "--test_data_path", TEST_DATA_PATH,
    "--output_dir", EVAL_OUTPUT_DIR,
    "--model_type", "task_prefix",
    "--batch_size", "64",
    "--max_input_length", str(MAX_INPUT_LENGTH),
    "--gen_max_len", str(GEN_MAX_LEN),
]

if USE_FP16 and not SANITY_RUN:
    eval_cmd.append("--fp16")

eval_cmd_str = " ".join(shlex.quote(part) for part in eval_cmd)
wrapped_cmd = f"set -o pipefail; {eval_cmd_str} 2>&1 | tee {shlex.quote(str(EVAL_LOG_PATH))}"
print("Running:", wrapped_cmd)
subprocess.run(["bash", "-lc", wrapped_cmd], check=True)


Running: set -o pipefail; python evaluate_test.py --model_path /kaggle/working/model_path/selected_thesis_prior_1 --test_data_path /kaggle/input/datasets/nightfury1103/anli-esnli-test/esnli_test.csv --output_dir artifacts/test_eval --model_type task_prefix --batch_size 64 --max_input_length 512 --gen_max_len 64 --fp16 2>&1 | tee artifacts/eval_run.log
/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plug

CompletedProcess(args=['bash', '-lc', 'set -o pipefail; python evaluate_test.py --model_path /kaggle/working/model_path/selected_thesis_prior_1 --test_data_path /kaggle/input/datasets/nightfury1103/anli-esnli-test/esnli_test.csv --output_dir artifacts/test_eval --model_type task_prefix --batch_size 64 --max_input_length 512 --gen_max_len 64 --fp16 2>&1 | tee artifacts/eval_run.log'], returncode=0)

In [11]:
predictions_path = Path(EVAL_OUTPUT_DIR) / f"{TRAIN_FILENAME}_test_predictions.csv"
metrics_path = Path(EVAL_OUTPUT_DIR) / f"{TRAIN_FILENAME}_test_metrics.json"

df = pd.read_csv(predictions_path)
with metrics_path.open() as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))
print("empty predictions:", int(df["prediction_is_empty"].sum()))
print("prediction preview:", df["prediction"].head().tolist())
print("label distribution:", df["label"].value_counts().to_dict())
print("top predictions:", df["prediction"].fillna("<empty>").value_counts().head(10).to_dict())


{
  "test_accuracy_exact": 0.0,
  "test_accuracy_contains_norm": 0.0,
  "test_accuracy_contains_raw_lower": 0.0,
  "num_examples": 9824,
  "prediction_seconds": 189.0739159719999,
  "seconds_per_example": 0.019246123368485333,
  "prediction_empty_ratio": 1.0,
  "fallback_report": [
    {
      "name": "single_gpu_full_precision",
      "empty_ratio": 1.0,
      "seconds": 188.49673004600004,
      "preview": [
        "",
        "",
        "",
        "",
        ""
      ]
    },
    {
      "name": "single_gpu_raw_input_full_precision",
      "empty_ratio": 1.0,
      "seconds": 189.99926760000017,
      "preview": [
        "",
        "",
        "",
        "",
        ""
      ]
    },
    {
      "name": "single_gpu_prefixed_input_full_precision",
      "empty_ratio": 1.0,
      "seconds": 187.15862523800024,
      "preview": [
        "",
        "",
        "",
        "",
        ""
      ]
    }
  ]
}
empty predictions: 9824
prediction preview: [nan, nan, nan, nan, nan]
la

In [12]:
display(df[["input", "label", "prediction", "prediction_is_empty"]].head())


,input,label,prediction,prediction_is_empty
0,This church choir sings to the masses as they ...,neutral,NaN,True
1,This church choir sings to the masses as they ...,entailment,NaN,True
2,This church choir sings to the masses as they ...,contradiction,NaN,True
3,"A woman with a green headscarf, blue shirt and...",neutral,NaN,True
4,"A woman with a green headscarf, blue shirt and...",entailment,NaN,True


In [13]:
df

,Unnamed: 0,label,llm_label,rationale,input,prediction,prediction_is_empty
0,0,neutral,neutral,The church choir singing to the masses does no...,This church choir sings to the masses as they ...,NaN,True
1,1,entailment,entailment,The church choir is singing joyous songs from ...,This church choir sings to the masses as they ...,NaN,True
2,2,contradiction,neutral,A choir singing at a baseball game is not nece...,This church choir sings to the masses as they ...,NaN,True
3,3,neutral,neutral,The woman is not necessarily young.,"A woman with a green headscarf, blue shirt and...",NaN,True
4,4,entailment,neutral,"The woman is smiling, but it is not necessaril...","A woman with a green headscarf, blue shirt and...",NaN,True
...,...,...,...,...,...,...,...
9819,9819,contradiction,contradiction,"If you're observing something, you cannot have...",Two women are observing something together.</s...,NaN,True
9820,9820,entailment,entailment,Women are girls.,Two women are observing something together.</s...,NaN,True
9821,9821,contradiction,contradiction,The man in the classroom is not flying a kite.,A man in a black leather jacket and a book in ...,NaN,True
9822,9822,entailment,entailment,A man in a black leather jacket and a book in ...,A man in a black leather jacket and a book in ...,NaN,True
